# Pretrained Models with Hugging Face

## Learning goals
- Understand what a pretrained model is.
- Load Hugging Face pipelines directly in Colab.
- Run text inference on simple examples.
- Interpret labels, scores, and tradeoffs.
- Connect this hands-on work to foundation models and practical evaluation.


In [ ]:
# Notebook-specific environment setup.
# We install missing packages so this notebook works on its own in Colab.

import importlib.util
import subprocess
import sys


REQUIRED_PACKAGES = {
    "transformers": "transformers>=4.40.0",
    "pandas": "pandas",
    "torch": "torch",
}

for import_name, pip_name in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
    else:
        print(f"{import_name} is already installed.")


## What does “pretrained” mean?

A pretrained model has already learned useful patterns from a large dataset before we use it.
Instead of training from scratch, we can reuse that earlier learning and run inference quickly.

**Why use Hugging Face?**

Hugging Face makes it easy to:
- load models
- compare tasks
- experiment with pipelines
- work with open models in a practical way


In [ ]:
# Import the pipeline API.
# A pipeline wraps tokenization, model loading, and post-processing into one simple object.

import pandas as pd
import torch

from transformers import pipeline


# device=0 means "use the first GPU" and device=-1 means "use CPU".
device = 0 if torch.cuda.is_available() else -1
print(f"Pipeline device: {'GPU' if device == 0 else 'CPU'}")


## Important lines in the previous cell

- `from transformers import pipeline`: imports the high-level API we will use.
- `device = 0 if torch.cuda.is_available() else -1`: selects GPU if one exists, otherwise CPU.
- A Hugging Face `pipeline` hides many steps that would otherwise take much more code.


## Example 1: sentiment analysis

This task predicts whether a piece of text sounds positive or negative.
It is a simple way to experience pretrained NLP without needing a training loop.


In [ ]:
# Load a sentiment analysis pipeline with an explicit model name.
# Using a named model makes the notebook more predictable and easier to teach.

sentiment_classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=device,
)

example_texts = [
    "This workshop was clear, practical, and easy to follow.",
    "The installation step was confusing and took too long.",
    "The notebook ran, but I still have mixed feelings about the results.",
    "I expected the model to fail, but it worked better than I thought.",
]

sentiment_results = sentiment_classifier(example_texts)

sentiment_df = pd.DataFrame(
    {
        "text": example_texts,
        "label": [result["label"] for result in sentiment_results],
        "score": [result["score"] for result in sentiment_results],
    }
)

display(sentiment_df.style.format({"score": "{:.3f}"}))


## What the sentiment pipeline lines do

- `pipeline("sentiment-analysis", ...)`: loads a ready-made text classification workflow.
- `model="distilbert-base-uncased-finetuned-sst-2-english"`: picks a specific pretrained model.
- `sentiment_classifier(example_texts)`: runs the model on a list of strings.
- Each returned dictionary includes a predicted `label` and a `score`.

**Important note**

The score is not a guarantee. It is only the model’s confidence-like output for that prediction.


## Example 2: question answering

This model answers questions using a short passage of context.
It is a useful contrast with sentiment analysis because it shows another pretrained task with the same API style.


In [ ]:
# Load a question-answering pipeline.
# The model searches for the answer span inside the context we provide.

question_answerer = pipeline(
    "question-answering",
    model="distilbert-base-cased-distilled-squad",
    device=device,
)

context = (
    "This workshop introduces supervised learning, train validation test splits, "
    "tabular modeling, convolutional neural networks, pretrained Hugging Face models, "
    "and an optional retrieval-augmented generation extension. "
    "The intended audience is non-expert to early-intermediate learners. "
    "The core workshop is designed to fit into about one hour."
)

questions = [
    "Who is the workshop designed for?",
    "How long is the core workshop?",
    "What optional extension is included?",
]

qa_rows = []

for question in questions:
    answer = question_answerer(question=question, context=context)
    qa_rows.append(
        {
            "question": question,
            "answer": answer["answer"],
            "score": answer["score"],
        }
    )

qa_df = pd.DataFrame(qa_rows)
display(qa_df.style.format({"score": "{:.3f}"}))


## Why this matters

With a few lines of code, we used two different pretrained NLP tasks.
That convenience is powerful, but it does not remove the need for:
- careful input design
- realistic evaluation
- attention to domain mismatch
- awareness of compute and deployment limits


In [ ]:
# Playground examples for experimentation.
# Replace these texts with product reviews, community messages, or your own examples.

playground_texts = [
    "The small local model was fast and private, but less polished.",
    "The larger hosted model gave better answers, but it cost more to use.",
    "This result sounds confident, yet I still want to verify it carefully.",
]

playground_results = sentiment_classifier(playground_texts)

playground_df = pd.DataFrame(
    {
        "text": playground_texts,
        "label": [result["label"] for result in playground_results],
        "score": [result["score"] for result in playground_results],
    }
)

display(playground_df.style.format({"score": "{:.3f}"}))


In [ ]:
# Playground for question answering.
# Edit the context or the questions and see how the answers change.

playground_context = (
    "Riverbend Library offers free beginner coding sessions every Wednesday evening. "
    "Participants can borrow laptops during class. "
    "Registration is encouraged, but walk-ins are welcome if seats remain."
)

playground_questions = [
    "When are the coding sessions held?",
    "Can participants borrow laptops?",
    "Is registration required?",
]

playground_qa_rows = []

for question in playground_questions:
    answer = question_answerer(question=question, context=playground_context)
    playground_qa_rows.append(
        {
            "question": question,
            "answer": answer["answer"],
            "score": answer["score"],
        }
    )

display(pd.DataFrame(playground_qa_rows).style.format({"score": "{:.3f}"}))


## Exercise prompts

- Try sentences with sarcasm or mixed sentiment. What happens?
- Change the QA context so some questions are **not** answerable. Does the output still sound confident?
- When is a pretrained model useful immediately?
- When might it fail because your domain is different from the training data?
- How does this connect to ideas like foundation models, multimodality, model size, and evaluation?

**Recap**

In this notebook, you used Hugging Face pipelines to run pretrained NLP models with very little code, while still thinking critically about quality, fit, and tradeoffs.
